# OH_3: Files & Modules — Practice Notebook

This notebook covers everything for this week's Office Hours: warm-up questions, debugging challenges, and 8 hands-on coding tasks (2 easy, 4 medium, 2 hard).

**Before you start:** make sure `sample_speech.txt` and `hacker_news_sample.txt` are saved in the **same folder** as this notebook — several tasks read them directly by filename.

**One Jupyter gotcha to know about:** several tasks use the `%%writefile` magic to create real `.py` module files on disk, which you then `import`. If you edit a module file and rewrite it, Python won't automatically notice the change on a second `import` (modules are cached). If your updated code doesn't seem to take effect, use `Kernel → Restart` and run the cells again from the top.

---
## Part 1: Warm-Up Questions

Discussion questions — no code needed. Think through your answer before reading the talking point.

### Files (1–5)

1. What's the difference between opening a file in `'r'`, `'w'`, `'x'`, and `'a'` mode? What happens to existing content in each case?
   *Talking point: `'w'` silently truncates the file — this trips people up constantly.*

2. Why is `with open("file.txt") as f:` preferred over calling `open()` and `close()` manually?
   *Talking point: guaranteed cleanup even if an exception is raised mid-read/write — segues nicely into context managers as a general concept.*

3. What's the difference between `.read()`, `.readline()`, and `.readlines()`? When would you reach for each one?
   *Talking point: memory usage on large files — `.read()`/`.readlines()` load everything at once, while iterating line-by-line (`for line in f:`) doesn't.*

4. If you open a file in text mode and it contains characters outside ASCII, what could go wrong, and what parameter fixes it?
   *Talking point: `encoding="utf-8"` — good excuse to mention this bites people the moment they leave English-only text data.*

5. What exception does Python raise if you try to read a file that doesn't exist? How would you handle it so the program doesn't crash?
   *Talking point: `FileNotFoundError`, and `try/except` vs. checking `os.path.exists()` first (race condition angle if you want to go deeper).*

### Modules (6–10)

6. What's the difference between `import module` and `from module import function`? What's a downside of `from module import *`?
   *Talking point: namespace pollution and losing track of where a name came from.*

7. What does `if __name__ == "__main__":` actually check, and why do we wrap script logic in it?
   *Talking point: a module's `__name__` is `"__main__"` only when run directly, not when imported — this is the #1 "click" moment for students.*

8. What makes a folder a Python **package** rather than just a folder of scripts?
   *Talking point: `__init__.py` (note: not strictly required since Python 3.3's namespace packages, but still the convention worth knowing).*

9. When you write `import mymodule`, how does Python know where to look for it?
   *Talking point: `sys.path` — current directory, installed packages, `PYTHONPATH`. Good moment to run `import sys; print(sys.path)` live.*

10. If you import the same module twice in a program, does its code run twice? Why or why not?
    *Talking point: `sys.modules` caching — imports after the first are just a lookup, not a re-execution.*

---
## Part 2: Debugging Challenges

For each challenge: run the code cell, read the traceback, and try to explain *why* it happens before reading the explanation below it.

### Challenge A — Mode mismatch

In [ ]:
f = open("scores.txt", "w")
data = f.read()

**Expected output:**
```
io.UnsupportedOperation: not readable
```
`'w'` opens the file for writing only — trying to `.read()` from it raises this error immediately.

### Challenge B — Using a file after the `with` block

In [ ]:
with open("sample_speech.txt") as f:
    first_line = f.readline()

second_line = f.readline()

**Expected output:**
```
ValueError: I/O operation on closed file.
```
The file is closed automatically the moment you exit the `with` block — `f` can't be used for reading afterward.

### Challenge C — Shadowing a standard library module

*(This one needs a real file to demonstrate — see Task 6 below for a live, runnable version of this exact bug.)*

```python
# file is named random.py, in the same folder
import random
print(random.randint(1, 10))
```

**What breaks?** → Python imports the local `random.py` instead of the standard library one, so `random.randint` doesn't exist → `AttributeError`. Never name your own files after stdlib modules (`random.py`, `math.py`, `json.py`, etc.).

### Challenge D — Reading twice without resetting position

In [ ]:
with open("sample_speech.txt") as f:
    content = f.read()
    content_again = f.read()
    print(len(content_again))

**Expected output:**
```
0
```
`content_again` is an empty string — the file pointer is already at the end after the first `.read()`. `f.seek(0)` would reset it back to the start.

### Challenge E — Relative path confusion

```python
# script located at /project/scripts/run.py
with open("data.txt") as f:
    ...
```

**What's the catch?** → `"data.txt"` is resolved relative to the **current working directory**, not the script's location — a classic source of "it works on my machine" bugs. `os.path.dirname(__file__)` or `pathlib.Path(__file__).parent` is the fix.

---
## Part 3: 8 Hands-On Tasks

**2 Easy · 4 Medium · 2 Hard.** Only Task 4 uses a JSON file — everything else that needs data uses plain `.txt`.

Cells marked `# TODO` are for you to fill in. Cells that create files use the `%%writefile` magic — running that cell writes an actual `.py` file into this notebook's folder.

### Task 1 (Easy): Build Your First Module
**Concept:** creating a module, `import`, `if __name__ == "__main__":`

Fill in the `TODO`s below. Running the cell will create a real `string_helpers.py` file in this notebook's folder.

In [ ]:
%%writefile string_helpers.py
def shout(text):
    # TODO: uppercase `text` and add "!!!" at the end, then return it
    pass

def count_vowels(text):
    # TODO: count and return how many vowels (a, e, i, o, u — any case) are in `text`
    pass

if __name__ == "__main__":
    print(shout("test"))
    print(count_vowels("test"))

Now import your module and use it:

In [ ]:
import string_helpers

print(string_helpers.shout("we did it"))
print(string_helpers.count_vowels("we did it"))

**Expected output:**
```
WE DID IT!!!
3
```

### Task 2 (Easy): Inventory Logger
**Concept:** `os` module, `datetime` module, appending to a `.txt` file

Fill in the TODOs, then **run this cell three times in a row** and check the file below.

In [ ]:
import os
from datetime import datetime

# TODO:
# 1. Open "inventory_log.txt" in append mode ("a") — this creates the file
#    automatically if it doesn't exist yet
# 2. Write one line containing the current timestamp (datetime.now()) and
#    the message " - Inventory checked", followed by a newline


In [ ]:
with open("inventory_log.txt") as f:
    print(f.read())

**Expected `inventory_log.txt` after 3 runs:**
```
2026-08-28 10:03:12 - Inventory checked
2026-08-28 10:03:14 - Inventory checked
2026-08-28 10:03:16 - Inventory checked
```
*(your timestamps will differ — that's expected)*

### Task 3 (Medium): Turn Old Code Into a Module
**Concept:** refactoring into a reusable module, files
**Data file needed:** `sample_speech.txt` (same folder as this notebook)

Fill in the TODOs in `text_utils.py` below.

In [ ]:
%%writefile text_utils.py
import re
from collections import Counter

def count_words(filename):
    # TODO: open the file, read it, and return the number of words (hint: .split())
    pass

def count_lines(filename):
    # TODO: open the file and return the number of lines
    pass

def most_common_words(filename, n):
    # TODO: read the file, lowercase it, extract words with re.findall(r"[a-z']+", text),
    # count them with Counter, and return the n most common as (word, count) tuples
    pass

In [ ]:
import text_utils

words = text_utils.count_words('sample_speech.txt')
lines = text_utils.count_lines('sample_speech.txt')
top5 = text_utils.most_common_words('sample_speech.txt', 5)

print(f"Words: {words}")
print(f"Lines: {lines}")
print(f"Top 5: {top5}")

**Expected output:**
```
Words: 138
Lines: 5
Top 5: [('the', 8), ('you', 6), ('is', 6), ('to', 4), ('and', 4)]
```

### Task 4 (Medium): Config Module → JSON File *(the one JSON task)*
**Concept:** modules as data containers, `json` module, files

In [ ]:
%%writefile config.py
settings = {
    "app_name": "OH Toolkit",
    "version": "1.0",
    "max_users": 50
}

Now import `config`, save its `settings` dict to `settings.json`, then load it back:

In [ ]:
import json
import config

# TODO: save config.settings to "settings.json" using json.dump(..., indent=4)


# TODO: reopen "settings.json" and load it back using json.load(), store it in `loaded`
loaded = None  # replace this

print(loaded["max_users"])

**Expected `settings.json`:**
```json
{
    "app_name": "OH Toolkit",
    "version": "1.0",
    "max_users": 50
}
```
**Expected printed output:** `50`

### Task 5 (Medium): Keyword Counter in a Text Log
**Concept:** reading a `.txt` file line by line, `os` module, folder creation
**Data file needed:** `hacker_news_sample.txt` (same folder as this notebook)

In [ ]:
import os

os.makedirs("reports", exist_ok=True)

python_count = 0
javascript_count = 0
java_only_count = 0

# TODO: open hacker_news_sample.txt and loop over its lines.
# For each line, case-insensitively:
#   - if it mentions "python", increment python_count
#   - if it mentions "javascript", increment javascript_count
#   - if it mentions "java" but NOT "javascript", increment java_only_count


# TODO: write the three counts to reports/summary.txt, one per line, e.g.
#   "Python mentions: 5"


In [ ]:
with open("reports/summary.txt") as f:
    print(f.read())

**Expected `reports/summary.txt`:**
```
Python mentions: 5
JavaScript mentions: 4
Java-only mentions: 4
```

### Task 6 (Medium): Shadowing Bug Hunt (debugging task)
**Concept:** module resolution, `sys.path`

These two files are given exactly as-is — the bug is already built in. Run both `%%writefile` cells, then run the one after that.

In [ ]:
%%writefile random.py
# A student's personal helper module — accidentally named the same
# as a standard library module.

quotes = [
    "Keep coding!",
    "You've got this.",
    "Debug it slowly, one line at a time."
]

def get_quote():
    return quotes[0]

In [ ]:
%%writefile main_broken.py
import random

number = random.randint(1, 10)
print("Your lucky number is:", number)

In [ ]:
!python3 main_broken.py

**Expected output when you run the cell above:**
```
Traceback (most recent call last):
  File "main_broken.py", line 3, in <module>
    number = random.randint(1, 10)
AttributeError: module 'random' has no attribute 'randint'
```

`import random` found the local `random.py` sitting right next to the script before it ever looked at the standard library, because the script's own folder is searched first. This is Warm-Up Question 9 showing up as a real bug.

**Your task:** fix it. Fill in the TODO below.

In [ ]:
# TODO: fix the bug.
# Easiest approach:
#   1. os.rename("random.py", "quotes_helper.py")
#   2. Rewrite main_broken.py so it imports "quotes_helper" instead of "random",
#      and uses the real `random` module for random.randint()
#   3. Re-run with !python3 main_broken.py to confirm it now works


### Task 7 (Hard): Build a Mini Package
**Concept:** packages, `__init__.py`, multiple import styles
**Data file needed:** `sample_speech.txt` (same folder as this notebook)

First, create the package folder:

In [ ]:
import os
os.makedirs("mypkg", exist_ok=True)

In [ ]:
%%writefile mypkg/__init__.py
# This file marks the `mypkg` folder as a Python package.

`file_ops.py` handles all file reading/writing:

In [ ]:
%%writefile mypkg/file_ops.py
def read_file(filename):
    # TODO: open the file and return its full text
    pass

def write_file(filename, text):
    # TODO: open the file in write mode and write `text` to it
    pass

`text_ops.py` only ever works with text — never touches a file directly:

In [ ]:
%%writefile mypkg/text_ops.py
import re
from collections import Counter

def word_count(text):
    # TODO: return the number of words in `text` (hint: .split())
    pass

def most_common_words(text, n):
    # TODO: lowercase `text`, extract words with re.findall(r"[a-z']+", text),
    # and return the n most common as (word, count) tuples
    pass

Now the driver — note the two different import styles:

In [ ]:
from mypkg import file_ops
from mypkg.text_ops import word_count
from mypkg import text_ops

text = file_ops.read_file('sample_speech.txt')
count = word_count(text)
top3 = text_ops.most_common_words(text, 3)

print(f"Word count: {count}")
print(f"Top 3 words: {top3}")

file_ops.write_file('summary.txt', f"Word count: {count}\nTop 3 words: {top3}\n")

**Expected console output:**
```
Word count: 138
Top 3 words: [('the', 8), ('you', 6), ('is', 6)]
```

*Talking point: why does `file_ops.py` need `text_ops.py` to stay file-agnostic? (Hint: it makes `text_ops` reusable on text that never came from a file — user input, an API response, etc.)*

### Task 8 (Hard): Three-Module Analysis Pipeline
**Concept:** splitting work across multiple modules, chaining them together, files
**Data file needed:** `sample_speech.txt` (same folder as this notebook)

Three separate modules, each with one job:

In [ ]:
%%writefile reader.py
def read_file(filename):
    # TODO: open the file and return its full text
    pass

In [ ]:
%%writefile analyzer.py
import re
from collections import Counter

def word_frequency(text, n=5):
    # TODO: lowercase `text`, extract words, count them, return the top n as
    # (word, count) tuples
    pass

def sentence_count(text):
    # TODO: split `text` on . ! ? (hint: re.split(r'[.!?]+', text)) and return
    # the number of non-empty sentences
    pass

In [ ]:
%%writefile writer.py
def save_report(filename, title, results):
    # TODO: write `title`, then a line of "=" the same length as `title`, then a
    # blank line, then each key/value pair from `results` as "label: value"
    pass

Now chain them together:

In [ ]:
import reader
import analyzer
import writer

text = reader.read_file('sample_speech.txt')
top_words = analyzer.word_frequency(text, 5)
sentences = analyzer.sentence_count(text)

results = {"Top 5 words": top_words, "Sentence count": sentences}
writer.save_report('report.txt', 'Speech Analysis Report', results)

with open('report.txt') as f:
    print(f.read())

**Expected `report.txt`:**
```
Speech Analysis Report
======================

Top 5 words: [('the', 8), ('you', 6), ('is', 6), ('to', 4), ('and', 4)]
Sentence count: 11
```

*Talking point: each module only knows about its own job — `reader` doesn't know what analysis will happen, `analyzer` never touches a file directly, `writer` doesn't care where the data came from. What would need to change if tomorrow the input came from a website instead of a `.txt` file? (Answer: only `reader.py`.)*

---
## Quick Reference

| Task | Difficulty | File type | Modules concept |
|---|---|---|---|
| 1. First Module | Easy | none | `import`, `__name__ == "__main__"` |
| 2. Inventory Logger | Easy | `.txt` | `os`, `datetime` |
| 3. Old Code → Module | Medium | `.txt` | custom module |
| 4. Config → JSON | Medium | `.json` | custom module + `json` |
| 5. Keyword Counter | Medium | `.txt` | `os` (folder creation) |
| 6. Shadowing Bug Hunt | Medium | none | module resolution / `sys.path` |
| 7. Mini Package | Hard | `.txt` | packages, `__init__.py` |
| 8. Analysis Pipeline | Hard | `.txt` | multi-module chaining |